In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")



In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(
    model="qwen3.5:latest",
    temperature=0
)

response = llm.invoke([
    HumanMessage(content="Explain AI agents simply")
])

print(response.content)

# **CHAIN WITH CUSTOM RUNNABLE**

In [3]:
# Task 1 [Prompt]
# Prompt
Prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie summarizer"),
        ("human", "Please summarize the movie in a brief: {input}"),
    ]
)

In [ ]:
# Task 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [5]:
# Task 3 [Parser]
str_parser = StrOutputParser()

In [7]:
# Task 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_parser(output: str) -> dict:
    """Parse a string output into a dictionary."""
    return {"text": output}

disctionary_parser_runnable = RunnableLambda(dictionary_parser) 


# **PARALLEL CHAIN 1**

In [ ]:
# Task 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a LinkedIn post generator"),
        ("human", "Create a post for the following text for LinkedIn: {text}"),
    ]
)

# Task 2 [LLM]
llm_ollama = ChatOllama(model="gpt-5-nano", temperature=0)

#Task 3 [Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_ollama | str_parser

# **PARALLEL CHAIN 2**

In [10]:
from langchain_core.runnables import RunnableParallel, RunnableLambda


In [ ]:
def insta_chain(text: dict):
    # Task 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a Instagram post generator"),
            ("human", "Create a post for the following text for Instagram: {text}"),
        ]
    )

    # Task 2 [LLM]
    llm_ollama = ChatOllama(model="gpt-5-nano", temperature=0)

    #Task 3 [Parser]
    str_parser = StrOutputParser()

    insta_chain = insta_prompt | llm_ollama | str_parser

    return insta_chain.invoke(text)

insta_chain_runable = RunnableLambda(insta_chain)

# **Final Orcheshtration**

In [ ]:
final_chain = (
        Prompt_template
        | llm_openai
        | str_parser
        | disctionary_parser_runnable
        | RunnableParallel(
            branches = {
                    "linkedin": chain_linkedin,
                    "instagram": insta_chain_runable
                }
            )
)

# final_chain_runnable = final_chain

In [13]:
def beautify(final_chain):
    linked_post = final_chain['branches']["linkedin"]
    insta_post = final_chain['branches']["instagram"]
    return {"linkedin": linked_post, "instagram": insta_post}

beautify_runnable = RunnableLambda(beautify)


In [14]:
# Task 4 [Chain]

beautify_final = final_chain | beautify_runnable